In [9]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR  # FIX 3: Added LR scheduler

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from torch.utils.data import random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


torch.manual_seed(42)
np.random.seed(42)

from google.colab import drive
drive.mount('/content/drive')


weights = ResNet50_Weights.IMAGENET1K_V1

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.RandomResizedCrop(
        224,
        scale=(0.85, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.95, 1.05)
    ),

    transforms.RandomPerspective(
        distortion_scale=0.1,
        p=0.2
    ),

    transforms.RandomGrayscale(p=0.05),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.0)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),

    transforms.RandomErasing(
        p=0.1,
        scale=(0.02, 0.08),
        ratio=(0.3, 3.3)
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset WITHOUT transforms first
full_dataset = datasets.ImageFolder(root="/content/drive/MyDrive/Project 4/dataset")
class_names = full_dataset.classes
num_classes = len(class_names)
# Split indices
dataset_size = len(full_dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_subset, val_subset, test_subset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)

# Apply different transforms
train_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(root="/content/drive/MyDrive/Project 4/dataset", transform=train_transform),
    train_subset.indices
)

val_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(root="/content/drive/MyDrive/Project 4/dataset", transform=val_transform),
    val_subset.indices
)

test_dataset = torch.utils.data.Subset(
    datasets.ImageFolder(root="/content/drive/MyDrive/Project 4/dataset", transform=test_transform),
    test_subset.indices
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)


def calculate_metrics(y_true, y_pred, class_names):
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    precision = report["weighted avg"]["precision"]
    recall    = report["weighted avg"]["recall"]
    f1        = report["weighted avg"]["f1-score"]
    return precision, recall, f1


def plot_confusion_matrix(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.close()
    print("Confusion matrix saved to confusion_matrix.png")

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Training Function
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    progress_bar = tqdm(loader, desc="Training")

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100*correct/total:.2f}%"})

    return running_loss / len(loader), 100 * correct / total

In [11]:
# Validation Function
def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        progress_bar = tqdm(loader, desc="Validation")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100*correct/total:.2f}%"})

    return running_loss / len(loader), 100 * correct / total


In [12]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=10, scheduler=None):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        print("\n" + "=" * 60)
        print(f"Epoch [{epoch+1}/{epochs}]")
        print("=" * 60)

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc   = validate(model, val_loader, criterion, device)

        if scheduler:
            scheduler.step()  # Step LR after each epoch

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.2f}%")
        print(f"Val Loss   : {val_loss:.4f}   | Val Acc   : {val_acc:.2f}%")
        print(f"LR         : {current_lr:.6f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_resnet50_transfer_learning.pth")
            print("Best model saved!")

    print("\nTraining Complete!")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    return history

In [13]:
# Evaluation Function
def evaluate_model(model, loader, criterion, class_names, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Testing"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_loss     = running_loss / len(loader)
    test_accuracy = 100 * correct / total
    precision, recall, f1 = calculate_metrics(all_labels, all_preds, class_names)

    print("\n" + "=" * 60)
    print("TEST RESULTS")
    print("=" * 60)
    print(f"Test Loss : {test_loss:.4f}")
    print(f"Accuracy  : {test_accuracy:.2f}%")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    plot_confusion_matrix(all_labels, all_preds, class_names)

In [14]:

# Training History Visualization
def plot_training_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"],   label="Validation Loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss Curve")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"],   label="Validation Accuracy")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy Curve")
    plt.legend()

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150)
    plt.close()
    print("Training history saved to training_history.png")

In [15]:
# Build Transfer Learning Model
model = resnet50(weights=weights)

# Phase 1: Freeze all backbone layers — only train the head
for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Phase 1: Train classifier head only
# CosineAnnealingLR scheduler added to improve convergence and generalization.
EPOCHS_PHASE1 = 10

criterion = nn.CrossEntropyLoss()
optimizer_phase1 = optim.Adam(model.fc.parameters(), lr=0.001)
scheduler_phase1 = CosineAnnealingLR(optimizer_phase1, T_max=EPOCHS_PHASE1, eta_min=1e-5)

print("\n" + "=" * 60)
print("PHASE 1: Training classifier head (backbone frozen)")
print("=" * 60)

history_phase1 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_phase1,
    device=device,
    epochs=EPOCHS_PHASE1,
    scheduler=scheduler_phase1,
)


PHASE 1: Training classifier head (backbone frozen)

Epoch [1/10]


Validation: 100%|██████████| 5/5 [00:34<00:00,  6.96s/it, Loss=1.2894, Acc=46.00%]


Train Loss : 1.3721 | Train Acc : 36.00%
Val Loss   : 1.2008   | Val Acc   : 46.00%
LR         : 0.000976
Best model saved!

Epoch [2/10]


Validation: 100%|██████████| 5/5 [00:01<00:00,  3.68it/s, Loss=1.2690, Acc=48.00%]


Train Loss : 1.2282 | Train Acc : 43.57%
Val Loss   : 1.1670   | Val Acc   : 48.00%
LR         : 0.000905
Best model saved!

Epoch [3/10]


...

Train Loss : 0.9837 | Train Acc : 60.43%
Val Loss   : 1.0962   | Val Acc   : 56.67%
LR         : 0.000105

Epoch [9/10]


Validation: 100%|██████████| 5/5 [00:01<00:00,  3.05it/s, Loss=1.3424, Acc=54.00%]


Train Loss : 0.9575 | Train Acc : 63.57%
Val Loss   : 1.0804   | Val Acc   : 54.00%
LR         : 0.000034

Epoch [10/10]


Validation: 100%|██████████| 5/5 [00:00<00:00,  5.30it/s, Loss=1.3437, Acc=55.33%]

Train Loss : 0.9764 | Train Acc : 60.14%
Val Loss   : 1.0817   | Val Acc   : 55.33%
LR         : 0.000010

Training Complete!
Best Validation Accuracy: 56.67%


In [16]:
plot_training_history(history_phase1)

Training history saved to training_history.png


In [17]:
# Phase 2 — Fine-tune backbone with lower LR
# Unfreeze all layers and train end-to-end at a reduced LR.
EPOCHS_PHASE2 = 5

for param in model.parameters():
    param.requires_grad = True

optimizer_phase2 = optim.Adam([
    {"params": model.fc.parameters(),  "lr": 1e-3},   # head: higher LR
    {"params": [p for name, p in model.named_parameters()
                if "fc" not in name],  "lr": 1e-4},   # backbone: lower LR
])
scheduler_phase2 = CosineAnnealingLR(optimizer_phase2, T_max=EPOCHS_PHASE2, eta_min=1e-6)

print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning full network (backbone unfrozen)")
print("=" * 60)

history_phase2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_phase2,
    device=device,
    epochs=EPOCHS_PHASE2,
    scheduler=scheduler_phase2,
)


PHASE 2: Fine-tuning full network (backbone unfrozen)

Epoch [1/5]


Validation: 100%|██████████| 5/5 [00:00<00:00,  5.31it/s, Loss=1.3383, Acc=62.67%]


Train Loss : 0.9386 | Train Acc : 62.43%
Val Loss   : 0.9942   | Val Acc   : 62.67%
LR         : 0.000905
Best model saved!

Epoch [2/5]


Validation: 100%|██████████| 5/5 [00:00<00:00,  5.24it/s, Loss=0.5677, Acc=80.00%]


Train Loss : 0.4541 | Train Acc : 84.71%
Val Loss   : 0.4223   | Val Acc   : 80.00%
LR         : 0.000655
Best model saved!

Epoch [3/5]


...

Train Loss : 0.2671 | Train Acc : 90.86%
Val Loss   : 0.4016   | Val Acc   : 86.67%
LR         : 0.000346
Best model saved!

Epoch [4/5]


Validation: 100%|██████████| 5/5 [00:00<00:00,  5.31it/s, Loss=0.3043, Acc=90.67%]


Train Loss : 0.1252 | Train Acc : 96.71%
Val Loss   : 0.2709   | Val Acc   : 90.67%
LR         : 0.000096
Best model saved!

Epoch [5/5]


Validation: 100%|██████████| 5/5 [00:00<00:00,  5.18it/s, Loss=0.2045, Acc=92.67%]


Train Loss : 0.0958 | Train Acc : 97.71%
Val Loss   : 0.2180   | Val Acc   : 92.67%
LR         : 0.000001
Best model saved!

Training Complete!
Best Validation Accuracy: 92.67%


In [18]:
plot_training_history(history_phase2)

Training history saved to training_history.png


In [19]:
# Load Best Model & Evaluate
model.load_state_dict(
    torch.load("best_resnet50_transfer_learning.pth", map_location=device)
)

evaluate_model(
    model=model,
    loader=test_loader,
    criterion=criterion,
    class_names=class_names,
    device=device,
)

Testing: 100%|██████████| 5/5 [00:31<00:00,  6.29s/it]



TEST RESULTS
Test Loss : 0.1937
Accuracy  : 94.00%
Precision : 0.9420
Recall    : 0.9400
F1-Score  : 0.9401

Classification Report:

              precision    recall  f1-score   support

       Angry       1.00      0.93      0.96        41
       Other       0.91      1.00      0.95        30
         Sad       0.91      0.93      0.92        44
       happy       0.94      0.91      0.93        35

    accuracy                           0.94       150
   macro avg       0.94      0.94      0.94       150
weighted avg       0.94      0.94      0.94       150

Confusion matrix saved to confusion_matrix.png
